# Week 1 Assignment 2 | Token Budget Calculator

**Objective:** Build a tool that accepts text input and returns:
- Token count (using GPT-4 tokenizer)
- Estimated API cost
- Context window usage percentage
- Recommendation for prompt optimization

---

## Step 1: Set Up tiktoken Library

First, install and import the `tiktoken` library (GPT-4 tokenizer)

In [ ]:
# Install tiktoken if not already installed
import subprocess
import sys

try:
    import tiktoken
    print("tiktoken already installed")
except ImportError:
    print("Installing tiktoken...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken"])
    import tiktoken

In [ ]:
import tiktoken

# Initialize the GPT-4 tokenizer
encoding = tiktoken.encoding_for_model("gpt-4")
print("GPT-4 tokenizer initialized successfully")

---

## Step 2: Define Configuration Variables

Set up pricing and context window parameters that are easy to update

In [ ]:
# === PRICING CONFIGURATION ===
# Update these values based on current provider pricing
# Example: OpenAI GPT-4 pricing (update with current rates)

PRICING_CONFIG = {
    "provider": "OpenAI",
    "model": "gpt-4",
    "input_price_per_1k_tokens": 0.03,      # $ per 1k input tokens
    "output_price_per_1k_tokens": 0.06,    # $ per 1k output tokens
    "context_window_size": 8192,            # Total context window tokens
}

# Display current configuration
print("\n=== Current Pricing Configuration ===")
for key, value in PRICING_CONFIG.items():
    print(f"{key}: {value}")

---

## Step 3: Create Token Counter Function

Function to compute exact token count using tiktoken

In [ ]:
def count_tokens(text):
    """
    Count the number of tokens in the given text using GPT-4 tokenizer.
    
    Args:
        text (str): The input text to tokenize
        
    Returns:
        int: Number of tokens
    """
    tokens = encoding.encode(text)
    return len(tokens)

# Test the function
test_text = "Hello, this is a test."
test_tokens = count_tokens(test_text)
print(f"Test text: '{test_text}'")
print(f"Token count: {test_tokens}")

---

## Step 4: Create Cost Estimation Function

In [ ]:
def estimate_api_cost(input_tokens, output_tokens=None):
    """
    Estimate the API cost for a given number of tokens.
    
    Args:
        input_tokens (int): Number of input tokens
        output_tokens (int): Estimated number of output tokens (default: same as input)
        
    Returns:
        dict: Cost breakdown
    """
    if output_tokens is None:
        output_tokens = input_tokens  # Assume output is same size as input
    
    input_cost = (input_tokens / 1000) * PRICING_CONFIG["input_price_per_1k_tokens"]
    output_cost = (output_tokens / 1000) * PRICING_CONFIG["output_price_per_1k_tokens"]
    total_cost = input_cost + output_cost
    
    return {
        "input_cost": round(input_cost, 6),
        "output_cost": round(output_cost, 6),
        "total_cost": round(total_cost, 6),
    }

# Test the function
test_cost = estimate_api_cost(100)
print("\nTest Cost Estimation (100 input tokens):")
for key, value in test_cost.items():
    print(f"  {key}: ${value}")

---

## Step 5: Create Context Window Usage Function

In [ ]:
def calculate_context_usage(input_tokens, output_tokens=None):
    """
    Calculate what percentage of the context window is consumed.
    
    Args:
        input_tokens (int): Number of input tokens
        output_tokens (int): Estimated number of output tokens
        
    Returns:
        dict: Context usage information
    """
    if output_tokens is None:
        output_tokens = input_tokens
    
    total_tokens_used = input_tokens + output_tokens
    context_window = PRICING_CONFIG["context_window_size"]
    usage_percentage = (total_tokens_used / context_window) * 100
    remaining_tokens = context_window - total_tokens_used
    
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens_used": total_tokens_used,
        "context_window_size": context_window,
        "usage_percentage": round(usage_percentage, 2),
        "remaining_tokens": remaining_tokens,
    }

# Test the function
test_usage = calculate_context_usage(1000, 500)
print("\nTest Context Window Usage (1000 input, 500 output tokens):")
for key, value in test_usage.items():
    print(f"  {key}: {value}")

---

## Step 6: Create Recommendation Function

In [ ]:
def make_recommendation(usage_data):
    """
    Provide a recommendation based on context window usage.
    
    Args:
        usage_data (dict): Output from calculate_context_usage()
        
    Returns:
        str: Recommendation message
    """
    usage_pct = usage_data["usage_percentage"]
    remaining = usage_data["remaining_tokens"]
    
    if usage_pct < 30:
        return (
            f"✅ PLENTY OF ROOM\n"
            f"You're using {usage_pct:.1f}% of the context window with {remaining} tokens remaining. "
            f"You can safely add more context or expect longer responses."
        )
    elif usage_pct < 70:
        return (
            f"⚠️  MODERATE USAGE\n"
            f"You're using {usage_pct:.1f}% of the context window with {remaining} tokens remaining. "
            f"Consider if you need all your input, or if you can trim the response expectations."
        )
    elif usage_pct < 90:
        return (
            f"⚠️⚠️ HIGH USAGE\n"
            f"You're using {usage_pct:.1f}% of the context window with only {remaining} tokens remaining. "
            f"The model has limited room for response. Consider trimming input or using a larger context window model."
        )
    else:
        return (
            f"❌ CRITICAL: EXCEEDS WINDOW\n"
            f"You're using {usage_pct:.1f}% of the context window. "
            f"Your input alone may exceed the available window. You MUST reduce input size significantly."
        )

# Test the function
print("\nTest Recommendations:")
print("\nScenario 1 (Low usage):")
test_rec1 = make_recommendation(calculate_context_usage(500, 500))
print(test_rec1)

print("\nScenario 2 (High usage):")
test_rec2 = make_recommendation(calculate_context_usage(5000, 2000))
print(test_rec2)

---

## Step 7: Create Main Calculator Function

Combine all functions into one easy-to-use tool

In [ ]:
def token_budget_calculator(text, estimated_output_tokens=None):
    """
    Complete token budget calculator.
    
    Args:
        text (str): Input text to analyze
        estimated_output_tokens (int): Estimated output tokens (default: same as input)
        
    Returns:
        dict: Complete analysis
    """
    # Count tokens
    input_tokens = count_tokens(text)
    
    if estimated_output_tokens is None:
        estimated_output_tokens = input_tokens
    
    # Calculate cost
    cost = estimate_api_cost(input_tokens, estimated_output_tokens)
    
    # Calculate context usage
    usage = calculate_context_usage(input_tokens, estimated_output_tokens)
    
    # Get recommendation
    recommendation = make_recommendation(usage)
    
    return {
        "input_text_length": len(text),
        "input_tokens": input_tokens,
        "estimated_output_tokens": estimated_output_tokens,
        "cost": cost,
        "context_usage": usage,
        "recommendation": recommendation,
    }

# Test the main function
test_input = """Artificial intelligence is transforming industries across the globe. 
From healthcare to finance, AI systems are being deployed to automate tasks, 
improve decision-making, and enhance user experiences. However, the rapid growth 
of AI also raises important questions about ethics, bias, and the future of work."""

result = token_budget_calculator(test_input)
print("\n" + "="*60)
print("TOKEN BUDGET CALCULATOR - TEST RESULTS")
print("="*60)
print(f"\nInput text length: {result['input_text_length']} characters")
print(f"Input tokens: {result['input_tokens']}")
print(f"Estimated output tokens: {result['estimated_output_tokens']}")
print(f"\nEstimated Cost:")
print(f"  Input cost: ${result['cost']['input_cost']}")
print(f"  Output cost: ${result['cost']['output_cost']}")
print(f"  Total cost: ${result['cost']['total_cost']}")
print(f"\nContext Window Usage: {result['context_usage']['usage_percentage']}%")
print(f"Remaining tokens: {result['context_usage']['remaining_tokens']}")
print(f"\n{result['recommendation']}")
print("\n" + "="*60)

---

## Step 8: Interactive Input

Now you can use the calculator with your own text!

In [ ]:
# Example 1: Analyze your own text
user_text = """
Enter your text here to analyze it.
"""

result = token_budget_calculator(user_text)

print("\n" + "="*60)
print("YOUR TEXT ANALYSIS")
print("="*60)
print(f"\nInput text length: {result['input_text_length']} characters")
print(f"Input tokens: {result['input_tokens']}")
print(f"Estimated output tokens: {result['estimated_output_tokens']}")
print(f"\nEstimated Cost (assume output = input size):")
print(f"  Input cost: ${result['cost']['input_cost']}")
print(f"  Output cost: ${result['cost']['output_cost']}")
print(f"  Total cost: ${result['cost']['total_cost']}")
print(f"\nContext Window Usage:")
print(f"  Used: {result['context_usage']['usage_percentage']}%")
print(f"  Remaining tokens: {result['context_usage']['remaining_tokens']}")
print(f"\n{result['recommendation']}")
print("\n" + "="*60)

---

## Step 9: Batch Analysis

Analyze multiple texts and compare costs

In [ ]:
# Compare different text examples
texts_to_compare = {
    "Short prompt": "What is machine learning?",
    "Medium prompt": "Explain how transformers work in natural language processing. Include details about attention mechanisms.",
    "Long prompt": user_text,  # Use your text from above
}

print("\n" + "="*60)
print("BATCH COMPARISON")
print("="*60)

for name, text in texts_to_compare.items():
    result = token_budget_calculator(text)
    print(f"\n{name}:")
    print(f"  Tokens: {result['input_tokens']}")
    print(f"  Total Cost: ${result['cost']['total_cost']}")
    print(f"  Context Usage: {result['context_usage']['usage_percentage']}%")

---

## Summary

### What This Calculator Does:

1. **Counts tokens** using the official GPT-4 tokenizer (tiktoken)
2. **Estimates API costs** based on current OpenAI pricing
3. **Calculates context window usage** as a percentage
4. **Provides recommendations** for prompt optimization

### How to Use:

- Update `PRICING_CONFIG` with current provider rates
- Replace the text in the "Interactive Input" cell with your own
- Run the cells to get instant cost and token estimates

### Key Insights:

- [Add your findings here about token usage patterns]
- [Document any interesting observations about input/output ratios]
- [Note how context window limits affect real-world applications]